# Environment setup
Create a Python virtual environment and install below packages

In [ ]:
%pip install -q -U --pre openvino openvino-tokenizers[transformers] openvino-genai --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install -q -U transformers "optimum-intel[openvino]"
%pip install -q -U --extra-index-url https://download.pytorch.org/whl/cpu  torch torchvision torchaudio
%pip install -q -U peft==0.17.1 librosa backoff

#All the models works well with transformers==4.57.6 which gets installed by default
#Phi4 and Phi3.5 requires transformers==4.51

In [ ]:
import os
import time
from pathlib import Path
import ipywidgets as widgets
import requests
import openvino_genai as ov_genai
from PIL import Image
import numpy as np
from openvino import Tensor

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

from notebook_utils import collect_telemetry
collect_telemetry("vlm.ipynb")

# Select device for inference

In [ ]:
from notebook_utils import device_widget
device = device_widget(default="NPU")
device

In [ ]:
## login to huggingfacehub to get access to pretrained model 

from huggingface_hub import notebook_login, whoami

try:
    whoami()
    print('Authorization token already provided')
except OSError:
    notebook_login()

# Select model
Pick a VLM from the dropdown

In [ ]:
model_ids = ["Qwen/Qwen2.5-VL-3B-Instruct", "microsoft/Phi-4-multimodal-instruct", "microsoft/Phi-3.5-vision-instruct", "google/gemma-3-4b-it", "openbmb/MiniCPM-V-2_6", "openbmb/MiniCPM-V-4_5"]

model_id = widgets.Dropdown(
    options=model_ids,
    description="Model:",
)
model_id


In [ ]:
# Ensure Phi-4/Phi-3.5 use transformers 4.51
if model_id.value in ("microsoft/Phi-4-multimodal-instruct", "microsoft/Phi-3.5-vision-instruct"):
    print("Installing transformers==4.51 for Phi-4/Phi-3.5...")
    get_ipython().run_line_magic("pip", "install -q -U transformers==4.51")

In [ ]:
additional_args_by_model = {
    "Qwen/Qwen2.5-VL-3B-Instruct": {
        "trust-remote-code": None,
        "sym": None,
        "group-size": "128",
        "weight-format": "int4",
    },
    "microsoft/Phi-4-multimodal-instruct": {
        "trust-remote-code": None,
        "sym": None,
        "group-size": "-1",
        "weight-format": "int4",
        "task": "image-text-to-text",
    },
    "microsoft/Phi-3.5-vision-instruct": {
        "trust-remote-code": None,
        "sym": None,
        "group-size": "-1",
        "weight-format": "int4",
    },
    "google/gemma-3-4b-it": {
        "trust-remote-code": None,
        "sym": None,
        "group-size": "-1",
        "weight-format": "int4",
    },
    "openbmb/MiniCPM-V-2_6": {
        "trust-remote-code": None,
        "sym": None,
        "group-size": "128",
        "weight-format": "int4",
    },
    "openbmb/MiniCPM-V-4_5": {
        "trust-remote-code": None,
        "sym": None,
        "group-size": "-1",
        "weight-format": "int4",
    },
}

additional_args = additional_args_by_model[model_id.value]

In [ ]:
print(f"Selected {model_id.value}")
pt_model_id = model_id.value
base_dir = pt_model_id.split("/")[-1]

quant = additional_args_by_model[model_id.value]["weight-format"]
group_size = additional_args_by_model[model_id.value]["group-size"]
group_label = "group128" if group_size == "128" else "group-1"
model_dir = Path(f"{base_dir}_{quant}_sym_{group_label}")

# Model Download
Download model from Huggingface using optimum-intel with corresponding parameters for that model

In [ ]:
from IPython.display import Markdown, display
from cmd_helper import optimum_cli

if model_dir.exists():
    display(Markdown(f"**Skipping export.** {model_dir} already exists."))
else:
    optimum_cli(pt_model_id, model_dir, additional_args=additional_args)
    display(Markdown(f"**Model downloaded at:** `{model_dir}`"))

# Inference
Download test image, specify prompt and do the inference and print the performance data

In [ ]:
#image_url = "https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/image/empty_road_mapillary.jpg"
image_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image_path = Path("demo.jpg")
if not image_path.exists():
    Image.open(requests.get(image_url, stream=True).raw).save(image_path)

question = "Explain the image in detail."
load_start = time.perf_counter()
#pipe = ov_genai.VLMPipeline(model_dir, device.value, config={"MAX_PROMPT_LEN": 4096})
pipe = ov_genai.VLMPipeline(model_dir, device.value)
load_time_s = time.perf_counter() - load_start

token_count = 0
first_token_time = None
last_token_time = None
token_latencies = []

display(Image.open(image_path))
print(f"Prompt: {question}")

def streamer(token: str):
    global token_count, first_token_time, last_token_time
    now = time.perf_counter()
    token_count += 1
    if first_token_time is None:
        first_token_time = now
    if last_token_time is not None:
        token_latencies.append(now - last_token_time)
    last_token_time = now
    print(token, end='', flush=True)
image_tensor = Tensor(np.array(Image.open(image_path).convert("RGB")))
gen_start = time.perf_counter()
result = pipe.generate(question, images=[image_tensor], max_new_tokens=1024, streamer=streamer)
gen_end = time.perf_counter()

print(f"\n\n=== Performance Metrics on {device.value} for model {model_dir} ===")
print(f"Model load time: {load_time_s:.3f}s")
print(f"Total tokens generated: {token_count}")

if first_token_time is not None:
    ttft = first_token_time - gen_start
    print(f"First token latency: {ttft:.3f}s")
    if len(token_latencies) > 0:
        avg_token_latency_ms = (sum(token_latencies) / len(token_latencies)) * 1000
        print(f"Avg token latency (after first): {avg_token_latency_ms:.2f} ms")
    if last_token_time is not None and token_count > 1:
        tps = (token_count - 1) / max(last_token_time - first_token_time, 1e-9)
        print(f"Tokens/sec (after first): {tps:.2f}")
else:
    print("No tokens were streamed; cannot compute TTFT/token latency.")
print(f"Total generation time: {gen_end - gen_start:.3f}s")